# Boltz-2: Binding Affinity **From a Provided Structure**

![Boltz2](https://img.shields.io/badge/Model-Boltz2-purple) ![Colab](https://img.shields.io/badge/Platform-Colab%20%7C%20Linux-lightgrey?logo=googlecolab) ![License](https://img.shields.io/badge/License-MIT-orange)

Feed **any structure (CIF, including single-chain or multimers)** and predict the
**binding affinity** of a ligand against it — with a one-click affinity toggle.

### How it works
Boltz-2 always runs its diffusion structure module; it doesn't score a pre-built
complex directly. So we feed *your* structure through Boltz-2's **`templates:`** block:
protein chains + sequences are auto-extracted from your CIF, your structure templates
those chains (optionally *forced* to restrain the backbone), and the ligand is given
as SMILES/CCD and flagged as the affinity **`binder`**.

### Affinity caveats (read once)
- Affinity = **one small-molecule ligand only** (no protein–protein, no multi-ligand).
- The affinity head does **not** explicitly model cofactors/ions/water/multimeric
  partners → for multimers, read it as *"ligand vs. the templated pocket"* and
  validate against MM/GBSA / ABFE / MST.
- Use **CIF** (PDB templates are buggy upstream); ligands with **≥128 atoms** are rejected for affinity.

**Runtime → Change runtime type → GPU** before you start.


In [ ]:
# @title 1. Install Boltz-2 + clone the repo
import sys, subprocess, threading, time, os, shutil, torch

class Color:
    CYAN="\033[96m"; GREEN="\033[92m"; YELLOW="\033[93m"; RED="\033[91m"; RESET="\033[0m"

print(f"{Color.CYAN}[i] Checking GPU availability...{Color.RESET}")
if not torch.cuda.is_available():
    print(f"{Color.RED}[\u2718] No GPU detected!{Color.RESET}")
    print(f"{Color.YELLOW}Runtime > Change runtime type > GPU (T4 or higher).{Color.RESET}")
else:
    print(f"{Color.GREEN}[\u2714] GPU detected:{Color.RESET} {torch.cuda.get_device_name(0)}")

REPO_URL  = "https://github.com/ahmedkoraby/boltz2-affinity.git"  # @param {type:"string"}
REPO_DIR  = "/content/" + REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
os.chdir("/content/")

def _spinner(msg, stop):
    syms = ["-", "\\", "|", "/"]; i = 0
    while not stop.is_set():
        sys.stdout.write(f"\r[{syms[i % 4]}] {msg}   "); sys.stdout.flush()
        time.sleep(0.1); i += 1
    sys.stdout.write("\r" + " " * (len(msg) + 12) + "\r")

def _run(cmd, msg, ok, fail, **kw):
    stop = threading.Event(); t = threading.Thread(target=_spinner, args=(msg, stop)); t.start()
    try:
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True, **kw)
        stop.set(); t.join(); print(ok)
    except Exception as e:
        stop.set(); t.join(); print(fail, e); raise

if os.path.isdir(REPO_DIR):
    print(f"{Color.YELLOW}[i] Repo exists, refreshing '{REPO_DIR}'...{Color.RESET}")
    shutil.rmtree(REPO_DIR)
_run(["git", "clone", REPO_URL, REPO_DIR],
     f"{Color.CYAN}Cloning {REPO_URL}...{Color.RESET}",
     f"[{Color.GREEN}\u2714{Color.RESET}] Repo cloned.",
     f"[{Color.RED}\u2718{Color.RESET}] Clone failed (is the repo public & pushed?).")

req = os.path.join(REPO_DIR, "requirements.txt")
pip_cmd = [sys.executable, "-m", "pip", "install", "-q"]
pip_cmd += (["-r", req] if os.path.exists(req) else ["boltz", "gemmi", "pyyaml", "py3Dmol"])
_run(pip_cmd,
     f"{Color.CYAN}Installing Boltz-2 + dependencies...{Color.RESET}",
     f"[{Color.GREEN}\u2714{Color.RESET}] Dependencies installed.",
     f"[{Color.RED}\u2718{Color.RESET}] Install failed.")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
from boltz_affinity import (parse_structure, summary, ProteinChain, Ligand, ensure_template_cif,
                            TemplateSpec, PocketConstraint, build_yaml_dict, write_yaml,
                            run_boltz, boltz_available, load_affinity, top_structure, report)
print(f"{Color.GREEN}All steps completed. boltz_affinity imported.{Color.RESET}")


In [ ]:
# @title 2. Download model weights + CCD (one-time warm-up)
# @markdown Runs a tiny prediction so Boltz-2 downloads its weights and the CCD
# @markdown component dictionary now (instead of mid-run). Slow the first time.
os.makedirs("/content/boltz_data", exist_ok=True)
with open("/content/boltz_data/_warmup.yaml", "w") as f:
    f.write("version: 1\n"
            "sequences:\n"
            "  - protein:\n      id: A\n      sequence: MVTPE\n"
            "  - ligand:\n      id: B\n      ccd: SAH\n")

_run(["boltz", "predict", "/content/boltz_data/_warmup.yaml",
      "--use_msa_server", "--out_dir", "/content/boltz_data/_warmup_out",
      "--accelerator", "gpu", "--devices", "1"],
     f"{Color.YELLOW}Downloading Boltz-2 weights + CCD dataset...{Color.RESET}",
     f"[{Color.GREEN}\u2714{Color.RESET}] Weights + CCD downloaded and validated.",
     f"[{Color.RED}\u2718{Color.RESET}] Warm-up failed.")


In [ ]:
# @title 3. Upload your structure (CIF recommended; single-chain or multimer)
from google.colab import files
os.makedirs("/content/work", exist_ok=True)
print("Select a .cif (or .pdb) file...")
up = files.upload()
fname = list(up.keys())[0]
STRUCTURE_PATH = "/content/work/" + fname
with open(STRUCTURE_PATH, "wb") as f:
    f.write(up[fname])

PARSED = parse_structure(STRUCTURE_PATH)
print("\n" + summary(PARSED))
PROTEIN_CHAINS = PARSED.protein_chains
if not PROTEIN_CHAINS:
    raise ValueError(
        "No protein chains detected. Affinity needs at least one protein chain "
        "plus a small-molecule ligand. Check your file has protein coordinates "
        "(re-export as standard PDB/mmCIF if it came from a docking tool).")
print(f"\n-> {len(PROTEIN_CHAINS)} protein chain(s) will be templated:",
      [c.chain_id for c in PROTEIN_CHAINS])


In [ ]:
# @title 4. Configure ligand + affinity options { run: "auto" }
# @markdown **Ligand** (the molecule whose affinity you want):
ligand_input_type = "smiles"  # @param ["smiles", "ccd"]
ligand_value = "N[C@@H](Cc1ccc(O)cc1)C(=O)O"  # @param {type:"string"}
ligand_id    = "L"  # @param {type:"string"}

# @markdown **Affinity** (one-click toggle):
predict_affinity = True  # @param {type:"boolean"}

# @markdown **Template** — use your uploaded structure to fix the protein pose:
use_structure_as_template = True  # @param {type:"boolean"}
force_backbone_to_template = True  # @param {type:"boolean"}
force_threshold_A = 5.0  # @param {type:"number"}

# @markdown **Optional pocket constraint** (recommended). Format `Chain:resi`, comma-separated,
# @markdown e.g. `A:34, A:56`. Leave blank to skip.
pocket_contacts = ""  # @param {type:"string"}

# @markdown **MSA**: let Boltz fetch MSAs automatically.
use_msa_server = True  # @param {type:"boolean"}

print("Ligand:", ligand_id, "=", ligand_value, f"({ligand_input_type})")
print("Affinity:", predict_affinity, "| Template:", use_structure_as_template,
      "| Force:", force_backbone_to_template, "| Pocket:", pocket_contacts or "(none)")


In [ ]:
# @title 5. Build the Boltz-2 input YAML
import yaml
proteins = [ProteinChain(c.chain_id, c.sequence) for c in PROTEIN_CHAINS]
used = {p.id for p in proteins}
lig_id = ligand_id if ligand_id not in used else next(x for x in "LMNOPQRSTUVWXYZ" if x not in used)
ligand = Ligand(lig_id,
                smiles=ligand_value if ligand_input_type == "smiles" else None,
                ccd=ligand_value if ligand_input_type == "ccd" else None)

templates = None
if use_structure_as_template:
    # Guarantee a Boltz-readable mmCIF template: returns the file as-is if gemmi
    # can read it, converts a PDB, or REBUILDS a clean CIF from _atom_site
    # coordinates when gemmi can't model-build the original.
    TEMPLATE_CIF = ensure_template_cif(STRUCTURE_PATH)
    if TEMPLATE_CIF != STRUCTURE_PATH:
        print(f"Template CIF prepared -> {TEMPLATE_CIF}")
    templates = [TemplateSpec(cif=TEMPLATE_CIF, chain_id=[p.id for p in proteins],
                              force=force_backbone_to_template,
                              threshold=force_threshold_A if force_backbone_to_template else None)]

pocket = None
if pocket_contacts.strip():
    contacts = []
    for tok in pocket_contacts.split(","):
        ch, ri = tok.strip().split(":"); contacts.append([ch.strip(), int(ri)])
    pocket = PocketConstraint(binder=lig_id, contacts=contacts)

data = build_yaml_dict(proteins, [ligand],
                       binder_id=lig_id if predict_affinity else None,
                       templates=templates, pocket=pocket)
YAML_PATH = "/content/work/input.yaml"
write_yaml(data, YAML_PATH)
print(open(YAML_PATH).read())


In [ ]:
# @title 6. Run Boltz-2
import os, shutil, glob
OUT_DIR = "/content/work/results"
shutil.rmtree(OUT_DIR, ignore_errors=True)   # clear stale/empty runs

RUN_OK = True
try:
    run_boltz(YAML_PATH, OUT_DIR, use_msa_server=use_msa_server,
              accelerator="gpu", stream=False)
except Exception as e:
    RUN_OK = False
    print(f"\n[!] boltz exited with an error: {e}")

# what got produced?
cifs    = glob.glob(os.path.join(OUT_DIR, "**", "*_model_*.cif"), recursive=True)
aff     = glob.glob(os.path.join(OUT_DIR, "**", "affinity_*.json"), recursive=True)
pre_aff = glob.glob(os.path.join(OUT_DIR, "**", "pre_affinity_*.npz"), recursive=True)

print("\n=== output tree under", OUT_DIR, "===")
files = [os.path.join(r, f) for r, _, fs in os.walk(OUT_DIR) for f in fs]
for p in files: print(p)
if not files: print("(empty)")

# Diagnose the classic affinity-cropper failure (missing pre_affinity npz)
if predict_affinity and not aff:
    if cifs and not pre_aff:
        print("""
[!] Structure was predicted but AFFINITY input (pre_affinity_*.npz) was never written.
    This is Boltz-2's affinity cropper skipping because it found NO protein-ligand
    contacts in the predicted complex (zero-size array). Fixes, in order:
      1) Add a POCKET constraint in cell 4 (pocket_contacts = "A:<resi>, A:<resi>, ...")
         using your known binding-site residues. This is the reliable fix.
      2) Turn ON use_structure_as_template so the real fold/pocket is used.
      3) Make sure the ligand is <=128 heavy+H atoms (ideally <=56).
      4) For a very large target, feed only the binding domain, not the full chain.
    The predicted STRUCTURE above is still usable; only affinity was skipped.""")
    elif not cifs:
        print("\n[!] No structure produced either. Check the boltz log above "
              "(common: CUDA out of memory on a large chain, or MSA-server failure).")


In [ ]:
# @title 7. Affinity dashboard
import matplotlib.pyplot as plt
if predict_affinity:
    res = load_affinity(OUT_DIR, binder_chain=lig_id)
    print(report(res))
    fig, ax = plt.subplots(1, 2, figsize=(8, 3))
    ax[0].bar(["P(binder)"], [res.probability_binary], color="#6c5ce7")
    ax[0].set_ylim(0, 1); ax[0].set_title("Binder probability")
    ax[1].bar(["pIC50"], [res.pic50], color="#00b894")
    ax[1].set_title("pIC50 (higher = tighter)")
    plt.tight_layout(); plt.show()
else:
    print("Affinity was disabled in step 4 (predict_affinity = False).")


In [ ]:
# @title 8. View the predicted 3D structure
import py3Dmol
s = top_structure(OUT_DIR)
print("Structure:", s)
view = py3Dmol.view(width=720, height=520)
view.addModel(open(s).read(), "cif")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.addStyle({"hetflag": True}, {"stick": {"colorscheme": "greenCarbon"}})
view.zoomTo(); view.show()


In [ ]:
# @title 9. Download all results (.zip)
import shutil
from google.colab import files
zip_path = "/content/boltz2_affinity_results"
shutil.make_archive(zip_path, "zip", OUT_DIR)
files.download(zip_path + ".zip")
print("Downloaded boltz2_affinity_results.zip")
